In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib

# Load data
df = pd.read_csv('data/denial_labels_train.csv')

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(max_features=1000), 'denial_text'),
        ('code', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['denial_code'])
    ])

# Encode labels
le = LabelEncoder()
y = le.fit_transform(df['label'])

# Split
X_train, X_test, y_train, y_test = train_test_split(df.drop('label', axis=1), y, test_size=0.2, random_state=42)

# Pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, multi_class='ovr'))
])

# Train
pipeline.fit(X_train, y_train)

# Predict & evaluate
y_pred = pipeline.predict(X_test)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Save
joblib.dump(pipeline, 'models/denial_classifier.pkl')
joblib.dump(le, 'models/label_encoder.pkl')


1.0
                   precision    recall  f1-score   support

  coding_bundling       1.00      1.00      1.00         4
      eligibility       1.00      1.00      1.00        17
medical_necessity       1.00      1.00      1.00        18
     missing_info       1.00      1.00      1.00        11
            other       1.00      1.00      1.00        13
    timely_filing       1.00      1.00      1.00        17
     underpayment       1.00      1.00      1.00         6

         accuracy                           1.00        86
        macro avg       1.00      1.00      1.00        86
     weighted avg       1.00      1.00      1.00        86



D:\Side_Projects\ERISA-AI-Portfolio-Challenge\env\lib\site-packages\sklearn\linear_model\_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


['label_encoder.pkl']

In [3]:
data="""{
    "patient_data": {
        "id": "PT-2026-001",
        "demographics": {
            "age": 45,
            "sex": "M",
            "location": "Carrollton, TX"
        },
        "scans": [
            {
                "modality": "CT",
                "series_id": "CT-ABD-20260324",
                "metadata": {
                    "voxel_spacing": [0.8, 0.8, 3.0],
                    "dimensions": [512, 512, 245],
                    "bit_depth": 16
                },
                "preprocessing": {
                    "steps": [
                        "window_level: center=40, width=400",
                        "resample: isotropic 1mm",
                        "normalize: z-score"
                    ],
                    "simpleitk_filters": {
                        "smoothing": "CurvatureFlowImageFilter(sigma=0.5)",
                        "segmentation": "MorphologicalWatershed"
                    }
                },
                "model_predictions": {
                    "cnn_3d": {
                        "architecture": "ResNet50-3D",
                        "input_shape": [1, 128, 128, 128],
                        "predictions": {
                            "lesion_prob": 0.87,
                            "confidence_map": {
                                "max_loc": [245, 312, 89],
                                "peak_value": 0.92
                            },
                            "class_probs": {
                                "benign": 0.13,
                                "malignant": 0.87
                            }
                        },
                        "metrics": {
                            "auc": 0.94,
                            "f1": 0.89,
                            "dice": 0.82
                        }
                    },
                    "transformer": {
                        "architecture": "ViT-3D-base",
                        "attention_heads": 12,
                        "layers": 6,
                        "predictions": {
                            "lesion_prob": 0.91,
                            "multi_scale_features": [
                                {"scale": "1/4", "confidence": 0.88},
                                {"scale": "1/8", "confidence": 0.93}
                            ]
                        }
                    }
                }
            },
            {
                "modality": "MRI",
                "series_id": "MRI-T1-20260324",
                "metadata": {
                    "voxel_spacing": [0.9, 0.9, 3.0],
                    "dimensions": [256, 256, 180]
                },
                "preprocessing": {
                    "steps": ["bias_correction", "skull_strip", "register_to_ct"]
                }
            }
        ]
    },
    "training_pipeline": {
        "pytorch_config": {
            "device": "cuda:0",
            "batch_size": 4,
            "optimizer": {
                "type": "AdamW",
                "lr": 1e-4,
                "weight_decay": 1e-5
            },
            "scheduler": "CosineAnnealingLR",
            "loss": {
                "primary": "DiceBCELoss",
                "auxiliary": "FocalLoss(alpha=0.25)"
            }
        },
        "data_augmentation": {
            "transforms": [
                "RandomRotation(15)",
                "RandomAffine(scale=0.1)",
                "ElasticTransform(alpha=120, sigma=12)",
                "GaussianNoise(std=0.01)"
            ]
        },
        "hardware": {
            "gpu": "NVIDIA A100 40GB",
            "raspberry_pi_setup": {
                "model": "Pi 5 8GB",
                "inference": "ONNX Runtime",
                "quantized": true
            }
        }
    },
    "experiment_tracking": {
        "run_id": "exp-20260324-v3",
        "wandb_project": "medical-imaging-segmentation",
        "hyperparams": {
            "lr": 1e-4,
            "dropout": 0.2,
            "batch_size": 4
        },
        "best_checkpoint": {
            "epoch": 47,
            "val_dice": 0.847,
            "path": "/models/best_resnet3d.pt"
        }
    }
}
"""

In [4]:
import json
from pprint import pprint
pprint({"name": "Ryan", "skills": ["PyTorch", "SimpleITK"], "active": True})

{'active': True, 'name': 'Ryan', 'skills': ['PyTorch', 'SimpleITK']}


In [5]:
item = json.loads(data)
print(item)

{'patient_data': {'id': 'PT-2026-001', 'demographics': {'age': 45, 'sex': 'M', 'location': 'Carrollton, TX'}, 'scans': [{'modality': 'CT', 'series_id': 'CT-ABD-20260324', 'metadata': {'voxel_spacing': [0.8, 0.8, 3.0], 'dimensions': [512, 512, 245], 'bit_depth': 16}, 'preprocessing': {'steps': ['window_level: center=40, width=400', 'resample: isotropic 1mm', 'normalize: z-score'], 'simpleitk_filters': {'smoothing': 'CurvatureFlowImageFilter(sigma=0.5)', 'segmentation': 'MorphologicalWatershed'}}, 'model_predictions': {'cnn_3d': {'architecture': 'ResNet50-3D', 'input_shape': [1, 128, 128, 128], 'predictions': {'lesion_prob': 0.87, 'confidence_map': {'max_loc': [245, 312, 89], 'peak_value': 0.92}, 'class_probs': {'benign': 0.13, 'malignant': 0.87}}, 'metrics': {'auc': 0.94, 'f1': 0.89, 'dice': 0.82}}, 'transformer': {'architecture': 'ViT-3D-base', 'attention_heads': 12, 'layers': 6, 'predictions': {'lesion_prob': 0.91, 'multi_scale_features': [{'scale': '1/4', 'confidence': 0.88}, {'scale

In [6]:
pprint(item)

{'experiment_tracking': {'best_checkpoint': {'epoch': 47,
                                             'path': '/models/best_resnet3d.pt',
                                             'val_dice': 0.847},
                         'hyperparams': {'batch_size': 4,
                                         'dropout': 0.2,
                                         'lr': 0.0001},
                         'run_id': 'exp-20260324-v3',
                         'wandb_project': 'medical-imaging-segmentation'},
 'patient_data': {'demographics': {'age': 45,
                                   'location': 'Carrollton, TX',
                                   'sex': 'M'},
                  'id': 'PT-2026-001',
                  'scans': [{'metadata': {'bit_depth': 16,
                                          'dimensions': [512, 512, 245],
                                          'voxel_spacing': [0.8, 0.8, 3.0]},
                             'modality': 'CT',
                             'model_predic